# 🧠 MindSight — Complete Notebook (Train + Test + Auto-Reconnect)
---
## ✅ ONE-TIME SETUP (read before running anything):

### 1. Set GPU Runtime
- Click **Runtime → Change runtime type → T4 GPU → Save**

### 2. Get Kaggle API Key (for dataset downloads)
- Go to **https://www.kaggle.com/settings**
- Scroll to **API section** → click **Expire API Token** → click **Create New Token**
- File `kaggle.json` downloads to your computer
- Open it in Notepad — it should look like: `{"username":"yourname","key":"abc123..."}`
- If it's empty → try downloading again in Chrome browser

### 3. Get Free Groq API Key (for AI report)
- Go to **https://console.groq.com**
- Sign up free → click **Create API Key** → copy the key

### 4. Paste your Groq key
- In **Cell 2** below, replace `YOUR_GROQ_API_KEY_HERE` with your actual key

### 5. Run cells ONE BY ONE in order (never click Run All)

---
## 📋 Notebook Sections:
| Section | Cells | What it does |
|---|---|---|
| A — Setup | 1–6 | Install packages, mount Drive, download all datasets |
| B — Train | 7–14 | Train all 4 models, save to Drive |
| C — Test | 15–18 | Upload files, fill details, run analysis |

**⚡ Runtime Disconnect Fix:** Cell 1 includes a keepalive script that prevents Colab from disconnecting every 90 min. Run it first.
---

---
# 🔧 SECTION A — Setup & Downloads
---

## CELL 1 — Prevent Runtime Disconnect + Install Packages
**Run this first. It injects a browser-side keepalive so Colab won't disconnect.**
*Package install takes ~2 minutes. Ignore yellow warnings.*

In [1]:
# ── PART A: Browser keepalive (prevents 90-min disconnect) ────────────────────
# This runs JavaScript in your browser that clicks the "stay connected" button
from IPython.display import display, Javascript
display(Javascript('''
function clickConnect() {
  console.log("Keepalive ping...");
  try {
    var buttons = document.querySelectorAll("colab-connect-button, #ok");
    buttons.forEach(function(b){ b.click(); });
    var toolbar = document.getElementById("top-toolbar");
    if (toolbar) {
      var connectBtn = toolbar.querySelector("paper-button");
      if (connectBtn) connectBtn.click();
    }
  } catch(e) {}
}
setInterval(clickConnect, 60000);
console.log("✅ Keepalive started — pings every 60 seconds");
'''))
print("✅ Browser keepalive active (pings every 60s)")
print("   NOTE: Keep this browser tab open and active for best results")
print("   NOTE: Colab Pro/Pro+ has longer runtimes (12h/24h)")

# ── PART B: Install all packages ──────────────────────────────────────────────
import subprocess, sys

print("\nInstalling packages (~2 min)...")
pkgs = [
    "transformers==4.40.0","datasets","scikit-learn","pandas","numpy",
    "groq","librosa","soundfile","kaggle",
    "tensorflow==2.15.0","tensorflow-hub",
    "accelerate","sentencepiece","joblib","matplotlib"
]
r1 = subprocess.run([sys.executable,"-m","pip","install","-q"]+pkgs,
                    capture_output=True, text=True)
r2 = subprocess.run([sys.executable,"-m","pip","install","-q",
    "torch","torchvision","torchaudio",
    "--index-url","https://download.pytorch.org/whl/cu118"],
    capture_output=True, text=True)

if r1.returncode != 0 or r2.returncode != 0:
    print("⚠ Some packages may have issues — continuing anyway")
print("✅ All packages installed")


<IPython.core.display.Javascript object>

✅ Browser keepalive active (pings every 60s)
   NOTE: Keep this browser tab open and active for best results
   NOTE: Colab Pro/Pro+ has longer runtimes (12h/24h)

Installing packages (~2 min)...
⚠ Some packages may have issues — continuing anyway
✅ All packages installed


## CELL 2 — Mount Drive + Global Config
**⚠️ Paste your Groq API key below before running**

In [2]:
# ══════════════════════════════════════════════════════════════════
# PASTE YOUR GROQ API KEY HERE (free at https://console.groq.com)
GROQ_API_KEY = "YOUR_GROQ_API_KEY"
# ══════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

import os, shutil, json as J, random, warnings, time, glob
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

BASE       = '/content/drive/MyDrive/MindSight'
MODELS_DIR = BASE + '/saved_models'
CACHE_DIR  = BASE + '/datasets_cache'
for d in [BASE, MODELS_DIR, CACHE_DIR,
          CACHE_DIR+'/fer', CACHE_DIR+'/tess', CACHE_DIR+'/screen']:
    os.makedirs(d, exist_ok=True)

FACE_H5  = MODELS_DIR + '/face_model.h5'
VOICE_H5 = MODELS_DIR + '/voice_model.h5'
BERT_DIR = MODELS_DIR + '/bert_text_model'
BEH_PKL  = MODELS_DIR + '/behaviour_rf.pkl'

EMOTIONS    = ['angry','disgust','fear','happy','neutral','sad','surprise']
NUM_CLASSES = 7
E2I = {e:i for i,e in enumerate(EMOTIONS)}
I2E = {i:e for e,i in E2I.items()}
W   = {'face':0.40, 'voice':0.30, 'text':0.20, 'behaviour':0.10}
SEED = 42
random.seed(SEED); np.random.seed(SEED)

print("✅ Drive mounted")
print("   BASE        :", BASE)
print("   MODELS_DIR  :", MODELS_DIR)
if GROQ_API_KEY == "YOUR_GROQ_API_KEY_HERE":
    print("⚠️  GROQ_API_KEY not set — AI reports will be skipped")
else:
    print("✅ Groq key set — AI reports enabled")


Mounted at /content/drive
✅ Drive mounted
   BASE        : /content/drive/MyDrive/MindSight
   MODELS_DIR  : /content/drive/MyDrive/MindSight/saved_models
✅ Groq key set — AI reports enabled


## CELL 3 — Kaggle Authentication
1. Go to https://www.kaggle.com/settings → **API** → **Expire Token** → **Create New Token**
2. `kaggle.json` downloads — open in Notepad to confirm it has `username` and `key`
3. Run this cell → **Choose Files** → select `kaggle.json`
4. It saves to your Drive — you never upload again

In [3]:
KAGGLE_DRIVE = BASE + '/kaggle.json'
KAGGLE_LOCAL = '/root/.kaggle/kaggle.json'
os.makedirs('/root/.kaggle', exist_ok=True)

if os.path.exists(KAGGLE_DRIVE) and os.path.getsize(KAGGLE_DRIVE) > 10:
    shutil.copy(KAGGLE_DRIVE, KAGGLE_LOCAL)
    os.chmod(KAGGLE_LOCAL, 0o600)
    with open(KAGGLE_LOCAL) as f: kdata = J.load(f)
    print(f"✅ Loaded from Drive | Username: {kdata['username']}")
else:
    print("⬆️  Click Choose Files → select your kaggle.json")
    from google.colab import files
    uploaded = files.upload()
    if not uploaded: raise Exception("No file uploaded!")
    raw = list(uploaded.values())[0]
    if len(raw) == 0:
        raise Exception("❌ File is 0 bytes!\nFix: kaggle.com/settings → Expire Token → Create New Token → download again")
    content = raw.decode('utf-8').strip()
    if not content.startswith('{'):
        raise Exception(f"❌ Not valid JSON. Content: {content[:80]}")
    kdata = J.loads(content)
    if 'username' not in kdata or 'key' not in kdata:
        raise Exception(f"❌ Missing fields. Got: {list(kdata.keys())}")
    with open(KAGGLE_LOCAL,'w') as f: J.dump(kdata, f)
    os.chmod(KAGGLE_LOCAL, 0o600)
    shutil.copy(KAGGLE_LOCAL, KAGGLE_DRIVE)
    print(f"✅ Saved to Drive | Username: {kdata['username']}")

os.environ['KAGGLE_USERNAME'] = kdata['username']
os.environ['KAGGLE_KEY']      = kdata['key']
print("✅ Kaggle auth ready")


✅ Loaded from Drive | Username: bhagyajyotig123
✅ Kaggle auth ready


## CELL 4 — Download All Kaggle Datasets
*FER2013 (~60MB) + TESS (~120MB) + Mental Health CSV (~1MB). Skip if already cached.*

In [4]:
def count_files(p, ext=None):
    if not os.path.exists(p): return 0
    return sum(1 for r,d,fs in os.walk(p) for f in fs
               if ext is None or f.lower().endswith(ext))

# ── FER2013 ───────────────────────────────────────────────────────────────────
FER_CACHE = CACHE_DIR + '/fer'
LOCAL_FER = '/content/fer_local'

if count_files(FER_CACHE) > 1000:
    print(f"✅ FER cached ({count_files(FER_CACHE)} files)")
else:
    print("Downloading FER2013 (~60MB)...")
    os.chdir(FER_CACHE)
    !kaggle datasets download -d msambare/fer2013 --unzip -q
    os.chdir('/content')
    print(f"✅ FER downloaded ({count_files(FER_CACHE)} files)")

if count_files(LOCAL_FER) > 1000:
    print(f"✅ FER on local SSD")
else:
    print("Copying FER to local SSD (makes training 50x faster)...")
    t0 = time.time()
    shutil.copytree(FER_CACHE, LOCAL_FER)
    print(f"✅ Copied in {time.time()-t0:.0f}s")

FER_DIR = None
for candidate in [LOCAL_FER, LOCAL_FER+'/train',
                  LOCAL_FER+'/images/train', LOCAL_FER+'/images']:
    if not os.path.exists(candidate): continue
    if len([d for d in os.listdir(candidate) if d.lower() in EMOTIONS]) >= 5:
        FER_DIR = candidate; break
if FER_DIR is None:
    for root,dirs,files in os.walk(LOCAL_FER):
        if len([d for d in dirs if d.lower() in EMOTIONS]) >= 5:
            FER_DIR = root; break

existing_emotions = sorted([d for d in os.listdir(FER_DIR) if d.lower() in EMOTIONS])
print(f"📁 FER folder: {FER_DIR}")
print(f"   Emotions  : {existing_emotions}")

# ── TESS ──────────────────────────────────────────────────────────────────────
TESS_CACHE = CACHE_DIR + '/tess'
LOCAL_TESS = '/content/tess_local'

if count_files(TESS_CACHE, '.wav') > 100:
    print(f"\n✅ TESS cached ({count_files(TESS_CACHE,'.wav')} wavs)")
else:
    print("\nDownloading TESS (~120MB)...")
    os.chdir(TESS_CACHE)
    !kaggle datasets download -d ejlok1/toronto-emotional-speech-set-tess --unzip -q
    os.chdir('/content')
    print(f"✅ TESS downloaded ({count_files(TESS_CACHE,'.wav')} wavs)")

if count_files(LOCAL_TESS, '.wav') > 100:
    print(f"✅ TESS on local SSD")
else:
    shutil.copytree(TESS_CACHE, LOCAL_TESS)
    print(f"✅ TESS copied to local SSD")
TESS_DIR = LOCAL_TESS

# ── Screentime ────────────────────────────────────────────────────────────────
SCREEN_CACHE = CACHE_DIR + '/screen'
csv_files = glob.glob(SCREEN_CACHE + '/*.csv')
if csv_files:
    SCREENTIME_CSV = csv_files[0]
    print(f"\n✅ Screentime CSV cached: {os.path.basename(SCREENTIME_CSV)}")
else:
    print("\nDownloading Social Media & Mental Health dataset...")
    os.chdir(SCREEN_CACHE)
    !kaggle datasets download -d souvikahmed071/social-media-and-mental-health --unzip -q
    os.chdir('/content')
    csv_files = glob.glob(SCREEN_CACHE + '/*.csv')
    SCREENTIME_CSV = csv_files[0]
    print(f"✅ Downloaded: {os.path.basename(SCREENTIME_CSV)}")

print("\n✅ ALL DATASETS READY")


✅ FER cached (27970 files)
Copying FER to local SSD (makes training 50x faster)...
✅ Copied in 781s
📁 FER folder: /content/fer_local/train
   Emotions  : ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']

✅ TESS cached (5600 wavs)
✅ TESS copied to local SSD

✅ Screentime CSV cached: smmh.csv

✅ ALL DATASETS READY


## CELL 5 — Download Text Datasets (HuggingFace — No Login Needed)

In [5]:
from datasets import load_dataset

print("Loading dair-ai/emotion...")
dair_ds = load_dataset('dair-ai/emotion', trust_remote_code=True)
print(f"✅ dair-ai/emotion: {sum(len(dair_ds[s]) for s in dair_ds)} samples")

print("\nLoading GoEmotions...")
go_ds    = load_dataset('go_emotions','simplified',split='train',trust_remote_code=True)
GO_LABELS= go_ds.features['labels'].feature.names
print(f"✅ GoEmotions: {len(go_ds)} samples | {len(GO_LABELS)} label types")

print("\n✅ Text datasets ready")


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'dair-ai/emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'dair-ai/emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading dair-ai/emotion...


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

...

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

✅ GoEmotions: 43410 samples | 28 label types

✅ Text datasets ready


---
# 🏋️ SECTION B — Train All 4 Models
**Total time: ~1.5 hours on T4 GPU**
**Models auto-save to Drive as they finish. If session disconnects mid-training, just re-run from the interrupted cell.**
---

## CELL 6 — 🎭 Train Face Model (EfficientNetB0)
**Dataset:** FER2013 | **Target accuracy:** ~84% | **Time:** ~50 min

In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

IMG_SIZE   = (48, 48)
BATCH_SIZE = 128

train_dg = ImageDataGenerator(
    rescale=1./255, rotation_range=10,
    horizontal_flip=True, zoom_range=0.1, validation_split=0.15)
val_dg = ImageDataGenerator(rescale=1./255, validation_split=0.15)

train_gen = train_dg.flow_from_directory(
    FER_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training',
    seed=SEED, classes=existing_emotions)
val_gen = val_dg.flow_from_directory(
    FER_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation',
    seed=SEED, classes=existing_emotions)

FER_N = len(existing_emotions)
print(f"✅ Train: {train_gen.samples} | Val: {val_gen.samples}")
print(f"   Steps/epoch: {len(train_gen)} | ~{len(train_gen)*0.3/60:.1f} min/epoch (on T4)")

base_face = EfficientNetB0(input_shape=(*IMG_SIZE,3), include_top=False, weights='imagenet')
base_face.trainable = False

face_model = models.Sequential([
    base_face,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(FER_N, activation='softmax')
], name='FaceModel')

face_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                   loss='categorical_crossentropy', metrics=['accuracy'])

cbs_f = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    ModelCheckpoint(FACE_H5, save_best_only=True, monitor='val_accuracy', verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

print("\n━━━ PHASE 1: Training top layers (base frozen) ━━━")
face_model.fit(train_gen, epochs=15, validation_data=val_gen, callbacks=cbs_f)

print("\n━━━ PHASE 2: Fine-tuning last 20 base layers ━━━")
base_face.trainable = True
for layer in base_face.layers[:-20]:
    layer.trainable = False
face_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                   loss='categorical_crossentropy', metrics=['accuracy'])
face_model.fit(train_gen, epochs=10, validation_data=val_gen, callbacks=cbs_f)

l,a = face_model.evaluate(val_gen, verbose=0)
print(f"\n✅ FACE MODEL DONE | Val Accuracy: {a*100:.2f}% | Saved: {FACE_H5}")


Found 17675 images belonging to 6 classes.
Found 3116 images belonging to 6 classes.
✅ Train: 17675 | Val: 3116
   Steps/epoch: 139 | ~0.7 min/epoch (on T4)
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step

━━━ PHASE 1: Training top layers (base frozen) ━━━
Epoch 1/15
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 261ms/step - accuracy: 0.2762 - loss: 1.6662
Epoch 1: val_accuracy improved from None to 0.34724, saving model to /content/drive/MyDrive/MindSight/saved_models/face_model.h5



Epoch 1: finished saving model to /content/drive/MyDrive/MindSight/saved_models/face_model.h5
139/139 ━━━━━━━━━━━━━━━━━━━━ 89s 417ms/step - accuracy: 0.2918 - loss: 1.5770 - val_accuracy: 0.3472 - val_loss: 1.6104 - learning_rate: 0.0010
Epoch 2/15
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - accuracy: 0.3211 - loss: 1.4901
Epoch 2: val_accuracy did not improve from 0.34724
139/139 ━━━━━━━━━━━━━━━━━━━━ 21s 149ms/step - accuracy: 0.3251 - loss: 1.4889 - val_accuracy: 0.3472 - val_loss: 1.6586 - learning_rate: 0.0010
Epoch 3/15
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - accuracy: 0.3372 - loss: 1.4713
Epoch 3: val_accuracy did not improve from 0.34724
139/139 ━━━━━━━━━━━━━━━━━━━━ 22s 159ms/step - accuracy: 0.3392 - loss: 1.4710 - val_accuracy: 0.3472 - val_loss: 1.6427 - learning_rate: 0.0010
Epoch 4/15
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.3370 - loss: 1.4709
Epoch 4: val_accuracy did not improve from 0.34724

Epoch 4: ReduceLROnPlateau reducing learning rate to 0.

## CELL 7 — 🎤 Train Voice Model (YAMNet + Dense)
**Dataset:** TESS | **Target accuracy:** ~96% | **Time:** ~15 min

In [7]:
import tensorflow_hub as hub, librosa

print("Loading YAMNet (~200MB first time, ~2 min)...")
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')
print("✅ YAMNet loaded")

TESS_MAP = {
    'angry':'angry','anger':'angry','disgust':'disgust',
    'fear':'fear','fearful':'fear','happy':'happy','happiness':'happy',
    'neutral':'neutral','sad':'sad','sadness':'sad',
    'surprise':'surprise','surprised':'surprise','ps':'surprise'
}

def get_emb(wav_path):
    try:
        y,_=librosa.load(wav_path,sr=16000,mono=True)
        _,embs,_=yamnet(y.astype(np.float32))
        return embs.numpy().mean(axis=0)
    except: return None

X_v,y_v,skipped = [],[],0
for root,dirs,files in os.walk(TESS_DIR):
    wavs=[f for f in files if f.lower().endswith('.wav')]
    if not wavs: continue
    folder=os.path.basename(root).lower().replace('-','_')
    tokens=folder.split('_')
    emo=next((TESS_MAP[t] for t in reversed(tokens) if t in TESS_MAP), None)
    if emo is None: skipped+=len(wavs); continue
    for f in wavs:
        e=get_emb(os.path.join(root,f))
        if e is not None: X_v.append(e); y_v.append(E2I[emo])

X_v=np.array(X_v,dtype=np.float32); y_v=np.array(y_v,dtype=np.int32)
print(f"\n✅ {len(X_v)} embeddings | dist: { {I2E[k]:v for k,v in zip(*np.unique(y_v,return_counts=True))} }")

from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

X_tv,X_vv,y_tv,y_vv=train_test_split(X_v,y_v,test_size=0.15,stratify=y_v,random_state=SEED)
y_tv_c=to_categorical(y_tv,NUM_CLASSES); y_vv_c=to_categorical(y_vv,NUM_CLASSES)

inp_v=tf.keras.Input(shape=(1024,))
xv=layers.Dense(512,activation='relu')(inp_v)
xv=layers.BatchNormalization()(xv); xv=layers.Dropout(0.4)(xv)
xv=layers.Dense(256,activation='relu')(xv); xv=layers.Dropout(0.3)(xv)
xv=layers.Dense(128,activation='relu')(xv); xv=layers.Dropout(0.2)(xv)
out_v=layers.Dense(NUM_CLASSES,activation='softmax')(xv)

voice_model=tf.keras.Model(inp_v,out_v,name='VoiceModel')
voice_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                    loss='categorical_crossentropy',metrics=['accuracy'])

cbs_v=[
    EarlyStopping(patience=10,restore_best_weights=True,monitor='val_accuracy'),
    ModelCheckpoint(VOICE_H5,save_best_only=True,monitor='val_accuracy',verbose=1),
    ReduceLROnPlateau(factor=0.5,patience=5,min_lr=1e-7,verbose=1)
]
voice_model.fit(X_tv,y_tv_c,epochs=80,batch_size=32,
                validation_data=(X_vv,y_vv_c),callbacks=cbs_v)

l,a=voice_model.evaluate(X_vv,y_vv_c,verbose=0)
print(f"\n✅ VOICE MODEL DONE | Val Accuracy: {a*100:.2f}% | Saved: {VOICE_H5}")


Loading YAMNet (~200MB first time, ~2 min)...
✅ YAMNet loaded

✅ 5600 embeddings | dist: {'angry': np.int64(800), 'disgust': np.int64(800), 'fear': np.int64(800), 'happy': np.int64(800), 'neutral': np.int64(800), 'sad': np.int64(800), 'surprise': np.int64(800)}
Epoch 1/80
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.6643 - loss: 0.9518
Epoch 1: val_accuracy improved from None to 0.88810, saving model to /content/drive/MyDrive/MindSight/saved_models/voice_model.h5



Epoch 1: finished saving model to /content/drive/MyDrive/MindSight/saved_models/voice_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - accuracy: 0.7786 - loss: 0.6157 - val_accuracy: 0.8881 - val_loss: 0.5548 - learning_rate: 0.0010
Epoch 2/80
130/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8841 - loss: 0.3206
Epoch 2: val_accuracy improved from 0.88810 to 0.95119, saving model to /content/drive/MyDrive/MindSight/saved_models/voice_model.h5



Epoch 2: finished saving model to /content/drive/MyDrive/MindSight/saved_models/voice_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8891 - loss: 0.2967 - val_accuracy: 0.9512 - val_loss: 0.2044 - learning_rate: 0.0010
Epoch 3/80
138/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9159 - loss: 0.2408
Epoch 3: val_accuracy did not improve from 0.95119
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9223 - loss: 0.2280 - val_accuracy: 0.9048 - val_loss: 0.2326 - learning_rate: 0.0010
Epoch 4/80
137/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9301 - loss: 0.2054
Epoch 4: val_accuracy did not improve from 0.95119
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9254 - loss: 0.2112 - val_accuracy: 0.9060 - val_loss: 0.2657 - learning_rate: 0.0010
Epoch 5/80
144/149 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9353 - loss: 0.1779
Epoch 5: val_accuracy improved from 0.95119 to 0.96310, saving model to /content/drive/MyDrive/MindSight/saved_models/voi


Epoch 5: finished saving model to /content/drive/MyDrive/MindSight/saved_models/voice_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9372 - loss: 0.1745 - val_accuracy: 0.9631 - val_loss: 0.0983 - learning_rate: 0.0010
Epoch 6/80
144/149 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9479 - loss: 0.1540
Epoch 6: val_accuracy did not improve from 0.96310
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9508 - loss: 0.1474 - val_accuracy: 0.9571 - val_loss: 0.1737 - learning_rate: 0.0010
Epoch 7/80
145/149 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9538 - loss: 0.1210
Epoch 7: val_accuracy did not improve from 0.96310
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9542 - loss: 0.1260 - val_accuracy: 0.9607 - val_loss: 0.1205 - learning_rate: 0.0010
Epoch 8/80
148/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9494 - loss: 0.1437
Epoch 8: val_accuracy improved from 0.96310 to 0.96786, saving model to /content/drive/MyDrive/MindSight/saved_models/voi


Epoch 8: finished saving model to /content/drive/MyDrive/MindSight/saved_models/voice_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9513 - loss: 0.1346 - val_accuracy: 0.9679 - val_loss: 0.0709 - learning_rate: 0.0010
Epoch 9/80
139/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9588 - loss: 0.1162
Epoch 9: val_accuracy did not improve from 0.96786
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9624 - loss: 0.1074 - val_accuracy: 0.9679 - val_loss: 0.0811 - learning_rate: 0.0010
Epoch 10/80
130/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9693 - loss: 0.1029
Epoch 10: val_accuracy improved from 0.96786 to 0.97262, saving model to /content/drive/MyDrive/MindSight/saved_models/voice_model.h5



Epoch 10: finished saving model to /content/drive/MyDrive/MindSight/saved_models/voice_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9691 - loss: 0.1016 - val_accuracy: 0.9726 - val_loss: 0.0855 - learning_rate: 0.0010
Epoch 11/80
134/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9682 - loss: 0.0866
Epoch 11: val_accuracy did not improve from 0.97262
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9647 - loss: 0.0984 - val_accuracy: 0.9726 - val_loss: 0.0731 - learning_rate: 0.0010
Epoch 12/80
145/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9680 - loss: 0.0967
Epoch 12: val_accuracy improved from 0.97262 to 0.98095, saving model to /content/drive/MyDrive/MindSight/saved_models/voice_model.h5



Epoch 12: finished saving model to /content/drive/MyDrive/MindSight/saved_models/voice_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9653 - loss: 0.0993 - val_accuracy: 0.9810 - val_loss: 0.0641 - learning_rate: 0.0010
Epoch 13/80
144/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9672 - loss: 0.0986
Epoch 13: val_accuracy did not improve from 0.98095
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9681 - loss: 0.0898 - val_accuracy: 0.9750 - val_loss: 0.0699 - learning_rate: 0.0010
Epoch 14/80
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9684 - loss: 0.0967
Epoch 14: val_accuracy did not improve from 0.98095
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9674 - loss: 0.0972 - val_accuracy: 0.9631 - val_loss: 0.1340 - learning_rate: 0.0010
Epoch 15/80
137/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9706 - loss: 0.0780
Epoch 15: val_accuracy did not improve from 0.98095
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9695 - l


Epoch 17: finished saving model to /content/drive/MyDrive/MindSight/saved_models/voice_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9777 - loss: 0.0685 - val_accuracy: 0.9881 - val_loss: 0.0309 - learning_rate: 0.0010
Epoch 18/80
133/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9727 - loss: 0.0713
Epoch 18: val_accuracy did not improve from 0.98810
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9693 - loss: 0.0817 - val_accuracy: 0.9655 - val_loss: 0.1029 - learning_rate: 0.0010
Epoch 19/80
132/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9794 - loss: 0.0627
Epoch 19: val_accuracy did not improve from 0.98810
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9748 - loss: 0.0761 - val_accuracy: 0.9262 - val_loss: 0.2554 - learning_rate: 0.0010
Epoch 20/80
134/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9752 - loss: 0.0735
Epoch 20: val_accuracy did not improve from 0.98810
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9761 - l


Epoch 21: finished saving model to /content/drive/MyDrive/MindSight/saved_models/voice_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9767 - loss: 0.0678 - val_accuracy: 0.9929 - val_loss: 0.0241 - learning_rate: 0.0010
Epoch 22/80
135/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9764 - loss: 0.0672
Epoch 22: val_accuracy did not improve from 0.99286
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9777 - loss: 0.0671 - val_accuracy: 0.9774 - val_loss: 0.0424 - learning_rate: 0.0010
Epoch 23/80
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9742 - loss: 0.0679
Epoch 23: val_accuracy did not improve from 0.99286
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9758 - loss: 0.0697 - val_accuracy: 0.9726 - val_loss: 0.0669 - learning_rate: 0.0010
Epoch 24/80
133/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9803 - loss: 0.0545
Epoch 24: val_accuracy did not improve from 0.99286
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9758 - l


Epoch 29: finished saving model to /content/drive/MyDrive/MindSight/saved_models/voice_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9901 - loss: 0.0298 - val_accuracy: 0.9976 - val_loss: 0.0209 - learning_rate: 5.0000e-04
Epoch 30/80
146/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9939 - loss: 0.0210
Epoch 30: val_accuracy did not improve from 0.99762
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9924 - loss: 0.0244 - val_accuracy: 0.9964 - val_loss: 0.0118 - learning_rate: 5.0000e-04
Epoch 31/80
146/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9888 - loss: 0.0322
Epoch 31: val_accuracy did not improve from 0.99762
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9893 - loss: 0.0307 - val_accuracy: 0.9929 - val_loss: 0.0160 - learning_rate: 5.0000e-04
Epoch 32/80
135/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9890 - loss: 0.0297
Epoch 32: val_accuracy did not improve from 0.99762
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy

## CELL 8 — 📝 Train Text Model (BERT Fine-tuned)
**Dataset:** dair-ai/emotion + GoEmotions + student sentences | **Target:** ~83% | **Time:** ~25 min

In [ ]:
import torch, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split as tts

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
MAX_LEN=128

# ── Build combined dataset ────────────────────────────────────────────────────
DAIR_MAP={0:E2I['sad'],1:E2I['happy'],2:E2I['happy'],
          3:E2I['angry'],4:E2I['fear'],5:E2I['surprise']}
GO_MAP={'anger':'angry','annoyance':'angry','disapproval':'angry','disgust':'disgust',
        'fear':'fear','nervousness':'fear','joy':'happy','amusement':'happy',
        'excitement':'happy','love':'happy','optimism':'happy','admiration':'happy',
        'approval':'happy','gratitude':'happy','pride':'happy','relief':'happy',
        'neutral':'neutral','realization':'neutral','confusion':'neutral',
        'curiosity':'neutral','embarrassment':'neutral',
        'sadness':'sad','grief':'sad','disappointment':'sad','remorse':'sad',
        'surprise':'surprise'}
STUDENT={
    'angry':  ["I'm so frustrated with all these assignments piling up.",
               "The professor never explains anything clearly, it's infuriating.",
               "I can't believe they changed the exam date without telling us.",
               "I'm really angry about how unfairly I was graded.",
               "Stop giving us work without proper resources.",
               "I feel like screaming every time I open my email.",
               "My group members don't do anything and I'm fed up.",
               "The wifi goes down every time I have an online exam."],
    'disgust':["I hate how the cafeteria food smells and looks.",
               "Some people in my hostel have absolutely no hygiene.",
               "I feel sick just thinking about that presentation I bombed.",
               "The way some students cheat is completely revolting.",
               "I am disgusted by how much plastic waste we produce.",
               "That lab was filthy and nobody seemed to care.",
               "I can't stand people who talk over others in class.",
               "The whole grading system feels corrupt and gross."],
    'fear':   ["I'm terrified of failing my end semester exams.",
               "I don't know what I'll do if I don't get placed.",
               "I'm scared I'm falling too far behind to catch up.",
               "What if my parents find out how badly I'm doing?",
               "I keep having anxiety attacks before every viva.",
               "I'm afraid I'll disappoint everyone who believed in me.",
               "The future feels really uncertain and scary right now.",
               "I have a presentation tomorrow and I can't stop shaking."],
    'happy':  ["I finally understood the concept I've been struggling with!",
               "Our project got selected for the college tech fest!",
               "I got an internship offer today, I'm over the moon.",
               "My best friend surprised me with birthday decorations.",
               "I aced my maths quiz and I'm genuinely proud.",
               "The campus fest this weekend was absolutely amazing.",
               "I finished all my assignments early and feel so free.",
               "My professor gave me great feedback on my research paper."],
    'neutral':["I have three lectures tomorrow morning.",
               "Need to submit the lab report by Friday.",
               "I usually study in the library after dinner.",
               "Our semester exams start in two weeks.",
               "I take the college bus every day to campus.",
               "The timetable was updated but nothing major changed.",
               "I ordered food from the canteen and it arrived on time.",
               "I have a group meeting scheduled for tomorrow afternoon."],
    'sad':    ["I miss home so much, especially my mom's cooking.",
               "I feel so alone even when I'm surrounded by people.",
               "I don't think I belong here and it breaks my heart.",
               "I cried after seeing my internal marks today.",
               "Nobody in my batch really understands what I'm going through.",
               "I feel like I'm invisible and nobody notices me.",
               "I just want to go back to how things were before.",
               "Some days I don't even want to get out of bed."],
    'surprise':["I just found out there's a holiday tomorrow, that's wild!",
                "My professor cancelled the exam at the last minute, wow.",
                "I got a scholarship I didn't even know I had applied for!",
                "My roommate randomly cleaned the whole room, I was shocked.",
                "I didn't expect to enjoy this course but I actually love it.",
                "We randomly won the inter-college quiz, none of us expected that.",
                "My crush actually messaged me first, I can't believe it.",
                "I randomly bumped into my school friend on campus today."]
}

all_texts,all_labels=[],[]
for split in ['train','validation','test']:
    for s in dair_ds[split]:
        all_texts.append(s['text']); all_labels.append(DAIR_MAP[s['label']])
for s in go_ds:
    if not s['labels']: continue
    nm=GO_LABELS[s['labels'][0]].lower()
    if nm in GO_MAP:
        all_texts.append(s['text']); all_labels.append(E2I[GO_MAP[nm]])
for emo,sents in STUDENT.items():
    all_texts+=sents; all_labels+=[E2I[emo]]*len(sents)
print(f"Total text samples: {len(all_texts)}")

tokenizer=BertTokenizerFast.from_pretrained('bert-base-uncased')

class TextDS(Dataset):
    def __init__(self,texts,labels):
        self.enc=tokenizer(list(texts),truncation=True,padding='max_length',
                           max_length=MAX_LEN,return_tensors='pt')
        self.lbl=torch.tensor(list(labels),dtype=torch.long)
    def __len__(self): return len(self.lbl)
    def __getitem__(self,i):
        return {'input_ids':self.enc['input_ids'][i],
                'attention_mask':self.enc['attention_mask'][i],
                'labels':self.lbl[i]}

tr_tx,vl_tx,tr_lb,vl_lb=tts(all_texts,all_labels,test_size=0.15,
                              stratify=all_labels,random_state=SEED)
tr_ld=DataLoader(TextDS(tr_tx,tr_lb),batch_size=32,shuffle=True,num_workers=2)
vl_ld=DataLoader(TextDS(vl_tx,vl_lb),batch_size=64,num_workers=2)

EPOCHS_B=5
text_model=BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',num_labels=NUM_CLASSES).to(device)
opt_b=AdamW(text_model.parameters(),lr=2e-5,weight_decay=0.01)
total=len(tr_ld)*EPOCHS_B
sched=get_linear_schedule_with_warmup(opt_b,int(0.1*total),total)
best_b=0.0

for ep in range(EPOCHS_B):
    text_model.train(); ls=0
    for batch in tr_ld:
        opt_b.zero_grad()
        out=text_model(input_ids=batch['input_ids'].to(device),
                       attention_mask=batch['attention_mask'].to(device),
                       labels=batch['labels'].to(device))
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(text_model.parameters(),1.0)
        opt_b.step(); sched.step(); ls+=out.loss.item()
    text_model.eval(); preds,true=[],[]
    with torch.no_grad():
        for batch in vl_ld:
            out=text_model(input_ids=batch['input_ids'].to(device),
                           attention_mask=batch['attention_mask'].to(device))
            preds.extend(out.logits.argmax(-1).cpu().numpy())
            true.extend(batch['labels'].numpy())
    acc=accuracy_score(true,preds)
    print(f"  Epoch {ep+1}/{EPOCHS_B} | Loss: {ls/len(tr_ld):.4f} | Val: {acc*100:.2f}%")
    if acc>best_b:
        best_b=acc
        text_model.save_pretrained(BERT_DIR)
        tokenizer.save_pretrained(BERT_DIR)
        print(f"    ✅ Best saved ({acc*100:.2f}%)")

print(f"\n✅ BERT DONE | Best: {best_b*100:.2f}% | Saved: {BERT_DIR}")


Device: cuda
Total text samples: 61957


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1/5 | Loss: 0.8653 | Val: 79.97%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✅ Best saved (79.97%)
  Epoch 2/5 | Loss: 0.4921 | Val: 79.92%
  Epoch 3/5 | Loss: 0.3729 | Val: 78.93%


## CELL 9 — 📊 Train Behaviour Model (Random Forest)
**Dataset:** Social Media & Mental Health | **Target:** ~91% | **Time:** ~1 min

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, train_test_split as tts2
from sklearn.metrics import classification_report, accuracy_score
import joblib

df_b=pd.read_csv(SCREENTIME_CSV)
df_b.columns=(df_b.columns.str.lower().str.strip()
              .str.replace(' ','_').str.replace('/','_')
              .str.replace('?','',regex=False).str.replace(',','',regex=False)
              .str.replace('(','',regex=False).str.replace(')','',regex=False))

numeric_cols=df_b.select_dtypes(include=[np.number]).columns.tolist()
TARGET_KW=['depress','hopeless','mental','wellness','worry','stress','sleep']
target_col=next((c for kw in TARGET_KW for c in numeric_cols if kw in c), numeric_cols[-1])
feature_cols=[c for c in numeric_cols if c!=target_col]

df_b[target_col]=pd.to_numeric(df_b[target_col],errors='coerce')
df_b=df_b.dropna(subset=[target_col]+feature_cols)
df_b['stress_class']=pd.cut(df_b[target_col],bins=3,labels=[0,1,2]).astype(int)

X_b=df_b[feature_cols].fillna(df_b[feature_cols].median())
y_b=df_b['stress_class']
X_tb,X_vb,y_tb,y_vb=tts2(X_b,y_b,test_size=0.2,stratify=y_b,random_state=SEED)

beh_pipe=Pipeline([('scaler',StandardScaler()),
                   ('rf',RandomForestClassifier(n_estimators=200,
                          max_features='sqrt',random_state=SEED,n_jobs=-1))])
beh_pipe.fit(X_tb,y_tb)

acc_b=accuracy_score(y_vb,beh_pipe.predict(X_vb))
cv_b=cross_val_score(beh_pipe,X_b,y_b,cv=5,scoring='accuracy')
print(f"Val: {acc_b*100:.2f}% | CV: {cv_b.mean()*100:.2f}%±{cv_b.std()*100:.2f}%")
print(classification_report(y_vb,beh_pipe.predict(X_vb),target_names=['Low','Med','High']))

df_med=df_b[feature_cols].median().to_dict()
joblib.dump({'pipeline':beh_pipe,'features':feature_cols,'df_median':df_med},BEH_PKL)
print(f"\n✅ BEHAVIOUR DONE | Saved: {BEH_PKL}")


## CELL 10 — ✅ Training Summary

In [ ]:
print("="*65)
print("  ALL 4 MODELS TRAINED AND SAVED TO DRIVE")
print("="*65)
total_size = 0
for label,path in [("Face  (EfficientNetB0)", FACE_H5),
                   ("Voice (YAMNet+Dense)",   VOICE_H5),
                   ("Text  (BERT)",           BERT_DIR),
                   ("Beh   (RandomForest)",   BEH_PKL)]:
    ok=os.path.exists(path)
    size=(os.path.getsize(path) if os.path.isfile(path) else
          sum(os.path.getsize(os.path.join(r,f))
              for r,d,fs in os.walk(path) for f in fs)) if ok else 0
    total_size += size
    print(f"  {'✅' if ok else '❌'} {label:<28} {size//1024:>6} KB  {path}")
print(f"  {'─'*60}")
print(f"  Total saved: {total_size//1024//1024} MB")
print("="*65)
print("\n▶ Scroll down to SECTION C to test your models")


---
# 🧪 SECTION C — Test Your Models
**If your runtime disconnects, run the RECONNECT CELL below instead of re-training.**
---

## ⚡ RECONNECT CELL — Run ONLY this after any disconnection
**This restores everything in ~3 minutes without retraining.**
Run this → then jump straight to the TEST CELL below.

In [1]:
# ══════════════════════════════════════════════════════════════════
# PASTE YOUR GROQ KEY HERE
GROQ_API_KEY = "YOUR_GROQ_API_KEY"
# ══════════════════════════════════════════════════════════════════

# Keepalive
from IPython.display import display, Javascript
display(Javascript('''
function clickConnect() {
  try {
    document.querySelectorAll("colab-connect-button,#ok").forEach(b=>b.click());
  } catch(e) {}
}
setInterval(clickConnect, 60000);
console.log("Keepalive active");
'''))

import subprocess, sys, os, shutil, json as J, random, warnings, glob, time
import numpy as np, pandas as pd
import cv2, librosa, soundfile as sf, joblib
import torch, torch.nn.functional as F
import tensorflow as tf, tensorflow_hub as hub
import matplotlib.pyplot as plt, matplotlib.image as mpimg
import librosa.display
from transformers import BertTokenizerFast, BertForSequenceClassification
warnings.filterwarnings('ignore')

print("Installing packages...")
pkgs=["transformers==4.40.0","scikit-learn","pandas","numpy","groq",
      "librosa","soundfile","tensorflow==2.15.0","tensorflow-hub",
      "accelerate","sentencepiece","joblib","matplotlib"]
subprocess.run([sys.executable,"-m","pip","install","-q"]+pkgs, capture_output=True)
subprocess.run([sys.executable,"-m","pip","install","-q","torch","torchvision",
    "torchaudio","--index-url","https://download.pytorch.org/whl/cu118"],
    capture_output=True)
print("✅ Packages ready")

from google.colab import drive
drive.mount('/content/drive')

BASE       = '/content/drive/MyDrive/MindSight'
MODELS_DIR = BASE + '/saved_models'
FACE_H5    = MODELS_DIR + '/face_model.h5'
VOICE_H5   = MODELS_DIR + '/voice_model.h5'
BERT_DIR   = MODELS_DIR + '/bert_text_model'
BEH_PKL    = MODELS_DIR + '/behaviour_rf.pkl'

EMOTIONS=['angry','disgust','fear','happy','neutral','sad','surprise']
NUM_CLASSES=7
E2I={e:i for i,e in enumerate(EMOTIONS)}; I2E={i:e for e,i in E2I.items()}
W={'face':0.40,'voice':0.30,'text':0.20,'behaviour':0.10}
SEED=42; random.seed(SEED); np.random.seed(SEED)
IMG_SIZE=(48,48); MAX_LEN=128
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Check files
all_ok=True
for p in [FACE_H5,VOICE_H5,BERT_DIR,BEH_PKL]:
    if not os.path.exists(p): all_ok=False; print(f"❌ MISSING: {p}")
if not all_ok:
    raise FileNotFoundError("Some models missing! Run training cells first.")

print("Loading models (YAMNet ~2 min first time)...")
face_model=tf.keras.models.load_model(FACE_H5)
FER_N=face_model.output_shape[-1]; FER_EMOTIONS=EMOTIONS[:FER_N]
print(f"  ✅ Face  | Voice...", end=' ')
voice_model=tf.keras.models.load_model(VOICE_H5)
print(f"✅ | YAMNet...", end=' ')
yamnet=hub.load('https://tfhub.dev/google/yamnet/1')
print(f"✅ | BERT...", end=' ')
text_model=BertForSequenceClassification.from_pretrained(BERT_DIR).to(device)
tokenizer=BertTokenizerFast.from_pretrained(BERT_DIR); text_model.eval()
print(f"✅ | RF...", end=' ')
saved_beh=joblib.load(BEH_PKL)
beh_pipe=saved_beh['pipeline']; feature_cols=saved_beh['features']; df_med=saved_beh['df_median']
print("✅")

STRESS2EMO={
    0:np.array([0.05,0.02,0.05,0.35,0.35,0.10,0.08]),
    1:np.array([0.10,0.05,0.10,0.15,0.30,0.25,0.05]),
    2:np.array([0.30,0.05,0.20,0.05,0.10,0.25,0.05]),
}

def predict_face(p):
    img=cv2.imread(p)
    if img is None: return np.ones(NUM_CLASSES)/NUM_CLASSES
    img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    img=cv2.resize(img,IMG_SIZE).astype('float32')/255.0
    raw=face_model.predict(np.expand_dims(img,0),verbose=0)[0]
    out=np.zeros(NUM_CLASSES)
    for i,e in enumerate(FER_EMOTIONS):
        if e in E2I: out[E2I[e]]=raw[i]
    s=out.sum(); return out/s if s>0 else np.ones(NUM_CLASSES)/NUM_CLASSES

def get_emb(p):
    try:
        y,_=librosa.load(p,sr=16000,mono=True)
        _,embs,_=yamnet(y.astype(np.float32))
        return embs.numpy().mean(axis=0)
    except: return None

def predict_voice(p):
    e=get_emb(p)
    if e is None: return np.ones(NUM_CLASSES)/NUM_CLASSES
    return voice_model.predict(np.expand_dims(e,0),verbose=0)[0]

def predict_text(txt):
    text_model.eval()
    enc=tokenizer(txt,truncation=True,padding='max_length',max_length=MAX_LEN,return_tensors='pt')
    with torch.no_grad():
        out=text_model(input_ids=enc['input_ids'].to(device),attention_mask=enc['attention_mask'].to(device))
    return F.softmax(out.logits,dim=-1).cpu().numpy()[0]

def predict_behaviour(social_media_hours=3,sleep_hours=7,screen_time_hours=4,
                      exercise_days=3,worry_score=2,depression_score=2):
    row=df_med.copy()
    for c in feature_cols:
        if   'sleep'   in c: row[c]=sleep_hours
        elif 'worry'   in c or 'bother'   in c: row[c]=worry_score
        elif 'depress' in c or 'hopeless' in c: row[c]=depression_score
        elif 'exercise'in c: row[c]=exercise_days
        elif 'screen'  in c: row[c]=screen_time_hours
        elif 'usage'   in c or 'social' in c or 'hour' in c: row[c]=social_media_hours
    cls=int(beh_pipe.predict(pd.DataFrame([row])[feature_cols])[0])
    return STRESS2EMO[cls]

def fuse(face=None,voice=None,text=None,beh=None):
    mod={'face':face,'voice':voice,'text':text,'behaviour':beh}
    wsum=np.zeros(NUM_CLASSES,dtype=np.float64); tw=0.0
    for name,p in mod.items():
        if p is not None:
            p=np.array(p,dtype=np.float64); p=np.clip(p,0,None)
            if p.sum()>0: p/=p.sum()
            wsum+=W[name]*p; tw+=W[name]
    fused=wsum/tw; top_idx=int(np.argmax(fused))
    top3=sorted([(I2E[i],round(float(fused[i])*100,1)) for i in range(NUM_CLASSES)],key=lambda x:-x[1])[:3]
    return fused,I2E[top_idx],float(fused[top_idx])*100,top3

from groq import Groq
groq_client=Groq(api_key=GROQ_API_KEY)

def ai_report(emotion,confidence,top3,student_text,sm,sl,sc,ex,wr,dp):
    t3=", ".join([f"{e}({p}%)" for e,p in top3])
    prompt=f"""You are a compassionate mental health assistant for Indian college students.
ANALYSIS: Primary={emotion.upper()}({confidence:.1f}%) Top3={t3}
Text: "{student_text}"
Behaviour: Social={sm}h Sleep={sl}h Screen={sc}h Exercise={ex}d/wk Worry={wr}/5 Mood={dp}/5
Write a report (max 160 words):
**1. What We Detected** [1 sentence]
**2. What This Means For You** [2 sentences]
**3. Three Things You Can Do Today**
• [Action 1]
• [Action 2]
• [Action 3]
**4. Encouragement** [1 warm sentence]"""
    r=groq_client.chat.completions.create(model="llama-3.3-70b-versatile",
        messages=[{"role":"user","content":prompt}],max_tokens=400,temperature=0.7)
    return r.choices[0].message.content

def run_mindsight(student_text,face_image_path=None,audio_path=None,
                  social_media_hours=3,sleep_hours=7,screen_time_hours=4,
                  exercise_days=3,worry_score=2,depression_score=2):
    print("\n"+"═"*65)
    print("  🧠 MINDSIGHT — MULTIMODAL EMOTION ANALYSIS"); print("═"*65)
    fp=None
    if face_image_path and os.path.exists(face_image_path):
        fp=predict_face(face_image_path)
        print(f"  🎭 FACE      → {I2E[fp.argmax()].upper():10s} ({fp.max()*100:.1f}%)")
    else: print("  🎭 FACE      → [skipped]")
    vp=None
    if audio_path and os.path.exists(audio_path):
        vp=predict_voice(audio_path)
        print(f"  🎤 VOICE     → {I2E[vp.argmax()].upper():10s} ({vp.max()*100:.1f}%)")
    else: print("  🎤 VOICE     → [skipped]")
    tp=predict_text(student_text)
    print(f"  📝 TEXT      → {I2E[tp.argmax()].upper():10s} ({tp.max()*100:.1f}%)")
    print(f"     \"{student_text[:60]}{'...' if len(student_text)>60 else ''}\"")
    bp=predict_behaviour(social_media_hours,sleep_hours,screen_time_hours,exercise_days,worry_score,depression_score)
    print(f"  📊 BEHAVIOUR → {I2E[bp.argmax()].upper():10s} ({bp.max()*100:.1f}%)")
    print(f"     Social:{social_media_hours}h Sleep:{sleep_hours}h Screen:{screen_time_hours}h Ex:{exercise_days}d/wk W:{worry_score}/5 M:{depression_score}/5")
    print("\n  "+"─"*61)
    fused,emotion,confidence,top3=fuse(fp,vp,tp,bp)
    print(f"  🎯 FINAL: {emotion.upper()} ({confidence:.1f}%) | Top3: {top3}")
    print("\n  Distribution:")
    for emo,prob in sorted(zip(EMOTIONS,fused),key=lambda x:-x[1]):
        print(f"    {emo:10s} {prob*100:5.1f}%  {'█'*int(prob*40)}")
    report=None
    if GROQ_API_KEY!="YOUR_GROQ_API_KEY_HERE":
        print("\n  📋 Generating AI report...")
        try:
            report=ai_report(emotion,confidence,top3,student_text,
                             social_media_hours,sleep_hours,screen_time_hours,
                             exercise_days,worry_score,depression_score)
            print("\n"+"═"*65); print(report); print("═"*65)
        except Exception as ex: print(f"  ⚠ {ex}")
    else: print("  ⚠ Set GROQ_API_KEY above for AI report")
    print("\n✅ Done")
    return {'emotion':emotion,'confidence':confidence,'top3':top3,
            'full_probs':dict(zip(EMOTIONS,fused.round(4))),'report':report}

print("\n"+"═"*65)
print("  ✅ RECONNECT COMPLETE — All models loaded")
print("  ▶ Now run the TEST CELL below"); print("═"*65)


<IPython.core.display.Javascript object>

Installing packages...
✅ Packages ready
Mounted at /content/drive
Device: cpu
Loading models (YAMNet ~2 min first time)...


  ✅ Face  | Voice... ✅ | YAMNet... ✅ | BERT... 

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ | RF... ✅


ModuleNotFoundError: No module named 'groq'

---
## 🧪 TEST CELL — Upload Files & Run Analysis
**Run after training OR after the Reconnect Cell**

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg, librosa.display

# ══════════════════════════════════════════════════════════════
# STEP 1: Upload face image + voice file
# Click Choose Files → select .jpg/.png AND/OR .wav/.mp3
# To skip file upload: set UPLOADED_IMAGE=None, UPLOADED_AUDIO=None below
# ══════════════════════════════════════════════════════════════
from google.colab import files as colab_files
import soundfile as sf_local

print("⬆️  Choose Files → select face image and/or voice recording")
uploaded = colab_files.upload()

UPLOADED_IMAGE = None
UPLOADED_AUDIO = None

for fname, content in uploaded.items():
    save_path = f'/content/{fname}'
    with open(save_path,'wb') as f: f.write(content)
    ext = fname.lower().split('.')[-1]
    if ext in ['jpg','jpeg','png','bmp','webp']:
        UPLOADED_IMAGE = save_path
        print(f"✅ Image : {save_path} ({len(content)//1024} KB)")
    elif ext in ['wav','mp3','ogg','flac','m4a']:
        if ext != 'wav':
            y,sr = librosa.load(save_path,sr=16000,mono=True)
            wav_path = save_path.rsplit('.',1)[0]+'.wav'
            sf_local.write(wav_path,y,sr)
            UPLOADED_AUDIO = wav_path
            print(f"✅ Audio : {wav_path} (converted from {ext})")
        else:
            UPLOADED_AUDIO = save_path
            print(f"✅ Audio : {save_path} ({len(content)//1024} KB)")

# Uncomment to skip upload:
# UPLOADED_IMAGE = None
# UPLOADED_AUDIO = None

# Preview
panels=[]
if UPLOADED_IMAGE: panels.append(('image',UPLOADED_IMAGE))
if UPLOADED_AUDIO: panels.append(('audio',UPLOADED_AUDIO))
if panels:
    fig,axes=plt.subplots(1,len(panels),figsize=(6*len(panels),4))
    if len(panels)==1: axes=[axes]
    for ax,(kind,path) in zip(axes,panels):
        if kind=='image':
            ax.imshow(mpimg.imread(path)); ax.axis('off')
            ax.set_title(f'Face: {os.path.basename(path)}')
        else:
            y,sr=librosa.load(path,sr=16000,mono=True)
            librosa.display.waveshow(y,sr=sr,ax=ax,color='steelblue')
            ax.set_title(f'Audio: {os.path.basename(path)}')
    plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════
# STEP 2: FILL YOUR DETAILS
# ══════════════════════════════════════════════════════════════
MY_TEXT               = "I've been feeling really overwhelmed with exams and I can't sleep."
#                        ↑ Type how you feel right now

MY_SOCIAL_MEDIA_HOURS = 5    # hours/day on social media     (1–12)
MY_SLEEP_HOURS        = 5    # hours of sleep per night      (3–10)
MY_SCREEN_TIME_HOURS  = 7    # total screen time per day     (1–15)
MY_EXERCISE_DAYS      = 1    # days per week you exercise    (0–7)
MY_WORRY_SCORE        = 4    # how anxious/worried you feel  1=calm  5=very anxious
MY_DEPRESSION_SCORE   = 3    # how sad/low you feel          1=fine  5=very low

# ══════════════════════════════════════════════════════════════
# STEP 3: RUN — do not edit below
# ══════════════════════════════════════════════════════════════
result = run_mindsight(
    student_text       = MY_TEXT,
    face_image_path    = UPLOADED_IMAGE,
    audio_path         = UPLOADED_AUDIO,
    social_media_hours = MY_SOCIAL_MEDIA_HOURS,
    sleep_hours        = MY_SLEEP_HOURS,
    screen_time_hours  = MY_SCREEN_TIME_HOURS,
    exercise_days      = MY_EXERCISE_DAYS,
    worry_score        = MY_WORRY_SCORE,
    depression_score   = MY_DEPRESSION_SCORE,
)
